In [64]:
from collections import Counter
import os 

import polars as pl
from tqdm.auto import tqdm
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.backend import BaseEmbedder
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI
import openai
from bertopic import BERTopic
import numpy as np
import heapq
import httpx
from sentence_transformers import SentenceTransformer

In [65]:
file_path = "20260201_20260302_spam_ie.xlsx"
save_folder = "exp1"
id_col_name = "id"
txt_col_name = "transcript"

In [ ]:
os.mkdir(save_folder)
df = pl.read_excel(file_path, columns=[id_col_name, txt_col_name])
df.head

In [ ]:
df.describe()

In [68]:
#Дубли id 
df.group_by(id_col_name).count().filter(
    pl.col("count") > 1
)

/var/folders/52/dvfz327j7j721ylccjd_sxhhm9qdvw/T/ipykernel_17385/484974998.py:2: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  df.group_by(id_col_name).count().filter(


id,count
str,u32


# BERTopic

In [ ]:
# Конфигурация
MODEL = "frida"
CERT_PATH = False  # или False если без SSL
BATCH_SIZE = 64

In [72]:
def get_embedding(text: str) -> list[float]:
    """Один текст → вектор [1536 float]."""
    with httpx.Client(verify=CERT_PATH, timeout=120) as client:
        r = client.post(
            f"{BASE_URL}/embeddings",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json={"model": MODEL, "input": text},
        )
        r.raise_for_status()
        return r.json()["data"][0]["embedding"]


def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    """Список текстов → список векторов."""
    with httpx.Client(verify=CERT_PATH, timeout=120) as client:
        r = client.post(
            f"{BASE_URL}/embeddings",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json={"model": MODEL, "input": texts},
        )
        r.raise_for_status()
        data = r.json()["data"]
        data.sort(key=lambda x: x["index"])
        return [item["embedding"] for item in data]
    

class MyApiBackend(BaseEmbedder):
    def __init__(self, batch_size=BATCH_SIZE):
        super().__init__()
        self.batch_size = batch_size

    def embed(self, documents, verbose=False):
        """
        BERTopic вызовет этот метод внутри себя.
        documents: список строк (list of str)
        return: numpy array (n_docs, embedding_dim)
        """
        all_embeddings = []
        
        # Генератор батчей
        iterator = range(0, len(documents), self.batch_size)
        if verbose:
            iterator = tqdm(iterator, desc="Embedding via API")

        for i in iterator:
            batch = documents[i : i + self.batch_size]
            
            # Твой код вызова API
            # ВАЖНО: убедись что documents[i] это строка, а не список
            # BERTopic иногда передает одиночные слова для representation
            
            try:
                # Твоя функция get_embeddings_batch (предполагаем она уже определена выше)
                # Она должна возвращать list[list[float]]
                batch_vecs = get_embeddings_batch(batch) 
                all_embeddings.extend(batch_vecs)
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                # Fallback или raise, зависит от логики
                raise e

        # Самое важное: превращаем в numpy array
        return np.array(all_embeddings)



In [73]:
def get_frida_emb(df, BATCH_SIZE=BATCH_SIZE, frac=1):
    emb = []
    sampled = (
        df
        .select(txt_col_name)
        .drop_nulls()
        .sample(fraction=frac, seed=42)
        .to_series()
    )

    successful_transcripts = []

    for i in tqdm(range(0, len(sampled), BATCH_SIZE)):
        batch = sampled[i:i+BATCH_SIZE].to_list()
        try:
            batch_embeddings = get_embeddings_batch(batch)
            emb.extend(batch_embeddings)
            successful_transcripts.extend(batch)  # сохраняем только успешные
        except Exception as e:
            print(f"Batch {i//BATCH_SIZE} sizes: {[len(t) for t in batch]}")  # длины элементов в батче
            print(f"Batch {i//BATCH_SIZE} failed: {e}")
            # можно пробовать обрабатывать по одному элементу, если хочется finer-grained
            for t in batch:
                try:
                    emb.append(get_embedding(t))
                    successful_transcripts.append(t)
                except Exception as e2:
                    print(f"Failed transcript removed: {t[:30]}..., error: {e2}")

    # Преобразуем успешные обратно в DataFrame, если нужно
    df_emb = pl.DataFrame({
        "text": successful_transcripts,
        "embedding": emb
    })
    df_emb.write_parquet(f"{save_folder}/frida_embeddings__{os.path.splitext(file_path)[0]}__{frac:.2f}.parquet")

    print(df_emb.shape)
    return df_emb

def get_berta_emb(df, model_path="/Users/seranovchinnikov/models/BERTA", frac=1):
    sampled = (
        df
        .select(txt_col_name)
        .drop_nulls()
        .sample(fraction=frac, seed=42)
        .to_series()
        .to_list()
    )

    embedding_model = SentenceTransformer(model_path, device="mps")
    emb = embedding_model.encode(sampled, show_progress_bar=True)
    df_emb = pl.DataFrame({
        "text": sampled,
        "embedding": emb
    })
    df_emb.write_parquet(f"{save_folder}/berta_embeddings__{os.path.splitext(file_path)[0]}__{frac:.2f}.parquet")

    print(df_emb.shape)
    return df_emb

In [75]:
df_emb = get_frida_emb(df)

100%|██████████| 759/759 [2:01:57<00:00,  9.64s/it]  


(48525, 2)


In [74]:
df_emb = get_berta_emb(df)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4756.91it/s]
Default prompt name is set to 'Classification'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.
Batches: 100%|██████████| 1517/1517 [03:36<00:00,  7.01it/s]


(48525, 2)


In [18]:
df_emb = pl.read_parquet("exp1/frida_embeddings__20260201_20260302_spam_ie__0.01.parquet")
df_emb.shape

(485, 2)

In [27]:
texts = df_emb["text"].to_list()
embeddings = np.vstack(df_emb["embedding"].to_list())
embeddings.shape

(485, 1536)

In [ ]:
# 1. Инициализируем bertopic
my_embedding_model = MyApiBackend(batch_size=BATCH_SIZE)

umap_model = UMAP(n_neighbors=40,
                    n_components=5, 
                    min_dist=0.1, 
                    metric='cosine', 
                    random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=60, 
                        min_samples=1, 
                        metric='euclidean', 
                        cluster_selection_method='eom', 
                        prediction_data=True)

vectorizer_model = CountVectorizer(min_df=5, 
                                   ngram_range=(1, 3))

keybert_model = KeyBERTInspired()

mmr_model = MaximalMarginalRelevance(diversity=0.3)

client = openai.OpenAI(
    base_url="
    http_client=httpx.Client(verify=False)
)
prompt = """
I have a topic that contains the following documents: 
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a very short (like one-to-five words) topic label in the following format (please, respond only in russian):
topic: <topic label>
"""
openai_model = OpenAI(
    client,
    model="data/Qwen3-30B-A3B-Instruct-2507/",
    generator_kwargs={'temperature': 0},
    delay_in_seconds=0, 
    chat=True,
    nr_docs=15,
    doc_length=150,
    tokenizer='char',
    prompt=prompt,
)


# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
    "OpenAI": openai_model,
}

In [34]:
def fit_bertopic(texts, embeddings):
  topic_model = BERTopic(
    language='russian',

    # Pipeline models
    embedding_model=my_embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,

    # Hyperparameters
    top_n_words=20,
    verbose=True,
  )

  topics, probs = topic_model.fit_transform(texts, np.array(embeddings))
  return topics, probs, topic_model

In [35]:
topics, probs, topic_model = fit_bertopic(texts, embeddings)

2026-03-06 18:01:01,944 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-06 18:01:03,118 - BERTopic - Dimensionality - Completed ✓
2026-03-06 18:01:03,118 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-06 18:01:03,123 - BERTopic - Cluster - Completed ✓
2026-03-06 18:01:03,124 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 2/2 [00:00<00:00,  3.33it/s]
2026-03-06 18:01:13,004 - BERTopic - Representation - Completed ✓


In [56]:
def set_topic_labels(topic_model):
    chatgpt_topic_labels = {topic: " | ".join(list(zip(*values))[0]) for topic, values in topic_model.topic_aspects_["OpenAI"].items()}
    chatgpt_topic_labels[-1] = "Outlier Topic"
    topic_model.set_topic_labels(chatgpt_topic_labels)

def print_topic_simple(topic_model):
    for log in zip(topic_model.get_topic_info()["Count"],
    topic_model.get_topic_info()["CustomName"],           
    topic_model.get_topic_info()["OpenAI"],
    topic_model.get_topic_info()["Representative_Docs"]):
        print(log)  

def get_summarize_prompt(topic_model, num_docs_per_topic=1, docs_elpsize=600):
    topic_info = topic_model.get_topic_info()
    all_topics_text = ""
    for count, custom_name, openai_label, rep_docs in zip(
            topic_info["Count"],
            topic_info["CustomName"],
            topic_info["OpenAI"],
            topic_info["Representative_Docs"]):
        
        docs_text = "\n".join(rep_docs[:num_docs_per_topic])  
        if custom_name != "Outlier Topic":
            all_topics_text += (
                f"Theme: {openai_label}, num files: {count})\n"
                f"representative doc:\n{docs_text[:docs_elpsize]}\n\n"
            )

    prompt = f"""
We have a set of customer call topics, each with its own representative documents and theme:

{all_topics_text}
Based on all the information provided, summarize the topics into 5–10 main overarching directions. 
For each main direction, provide a very short label (1–5 words) in Russian that captures the essence of the group.
If multiple topics are similar, merge them into a single direction.
    """
    return prompt

def print_topics_summarize(client, topic_model):
        # Запрос к модели
    prompt = get_summarize_prompt(topic_model)
    print(prompt)
    print("==="*30)
    print("MODEL RESPONSE")
    response = client.chat.completions.create(
        model="data/Qwen3-30B-A3B-Instruct-2507/",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    # Вывод результатов
    print(response.choices[0].message.content)

def reduce_outlets(topic_model, texts, topics, embeddings, strategy="embeddings", threshold=0.1):
    new_topics = topic_model.reduce_outliers(
        texts, 
        topics, 
        embeddings=embeddings,
        strategy=strategy,# или "embeddings" (дольше, но точнее)
        threshold=threshold       # если уверенность ниже 10%, оставляем -1
    )

    # Обновите модель новыми топиками
    topic_model.update_topics(texts, topics=new_topics)

    # Проверьте, сколько теперь -1
    print(f"Старое кол-во аутлайеров: {topics.count(-1)}")
    print(f"Новое кол-во аутлайеров: {new_topics.count(-1)}")

In [ ]:
set_topic_labels(topic_model)
print("topics count: ", len(topic_model.get_topic_info()))
topic_model.get_topic_info()

In [ ]:
print_topic_simple(topic_model)

In [ ]:
topic_model.get_topic_info()[['Topic', 'Count', 'CustomName', 'KeyBERT', 'Representative_Docs']].to_excel('topics.xlsx')

In [ ]:
topics = topic_model.topics_
texts[np.array(topics) == -1]

outlier_texts = texts[np.array(topics) == -1]
outlier_emb = embeddings[np.array(topics) == -1]

In [ ]:
print_topics_summarize(client, topic_model)

In [ ]:
topic_model.visualize_documents(texts, embeddings=embeddings, custom_labels=True)


In [57]:
reduce_outlets(topic_model, texts, topics, embeddings)

ValueError: No outliers to reduce.

In [54]:
topic_model.visualize_hierarchy(custom_labels=True)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

texts = [
    "ассистент не может",
    "у ассистента не получилось",
    "ассистент не справился",
    "ассистент ограничен",
    "ассистент не смог", "модель не работает", "модель не может, попробуй снова", "объясни почему не смог", "ассистент не понял"
]
neg_emb = get_embeddings_batch(texts)
neg_emb = np.array(neg_emb)
neg_emb = np.mean(neg_emb, axis=0)

# Делаем reshape и сохраняем в ту же или новую переменную
if neg_emb.ndim == 1:
    neg_emb_2d = neg_emb.reshape(1, -1)
else:
    neg_emb_2d = neg_emb

# Теперь передаем neg_emb_2d (который точно 2D)
similarities = cosine_similarity(neg_emb_2d, np.array(emb))

# Дальше ваш код...
similarities = similarities.flatten()

df_results = pl.DataFrame({
    'text': abstracts,  # ваши тексты
    'similarity': similarities
})

top_results = df_results.sort_values(by='similarity', ascending=False).head(200)

with pd.option_context('display.max_colwidth', None):
    display(top_results[150:])